Do some preliminary analysis on the outputs.

**Note**: All the citation analysis here are localized within our dataset, which GPT assigns non-trivial opinion scores.

In [15]:
import pandas as pd
import numpy as np

dat = pd.read_csv('../data/SC_related_cites_010_GPT_processed.csv')

mechanism_keys = [
    'pure el-ph coupling',
    'bipolaron el-ph coupling',
    'AFM fluctuation',
    'FM fluctuation',
    'charge density wave',
    'nematic fluctuation',
    'plasmon fluctuation',
    'pure el correlation',
    'spin liquid el correlation'
]

# Parse the dict from string
dat['parsed'] = dat['GPT_output'].apply(eval)

# Keep rows where ANY mechanism ≠ 0
mask = dat['parsed'].apply(
    lambda d: any(d[k] != 0 for k in mechanism_keys)
)

df = dat[mask]
print('Valid entries:', len(df))
df[df['id'] == 'pub.1060437772']

Valid entries: 5740


,abstract,concepts_scores,doi,id,journal,times_cited,title,year,GPT_output,parsed
336,The superconducting transition temperature is ...,[{'concept': 'band-structure density of states...,10.1103/physrev.167.331,pub.1060437772,"{'id': 'jour.1312024', 'title': 'Physical Revi...",5330,Transition Temperature of Strong-Coupled Super...,1968,"{""system"":""various metals and alloys"",""model"":...","{'system': 'various metals and alloys', 'model..."


Get citation graph

In [ ]:
import dimcli
import pandas as pd
import time
import math

# =====================================================
# LOAD PAPER IDS
# =====================================================

paper_ids = df["id"].dropna().unique().tolist()
paper_set = set(paper_ids)
id2year = dict(zip(df["id"], df["year"]))
print(f"{len(paper_ids)} ids loaded")

# =====================================================
# LOGIN
# =====================================================

from utils.DIMCLI_APIKEY import API_KEY

dimcli.login(key=API_KEY)
dsl = dimcli.Dsl()

# =====================================================
# QUERY FUNCTION
# =====================================================

def fetch_refs_years(batch):
    id_list = ", ".join(f'"{pid}"' for pid in batch)
    q = f"""
    search publications
        where id in [{id_list}]
    return publications[id + reference_ids + year]
    """
    return dsl.query_iterative(q, limit=400, pause=0.5).as_dataframe()

# =====================================================
# BUILD INTERNAL GRAPH
# =====================================================

edges = []
neighbors = {}   # neighbors[src] = [ [id, year], ... ]

batch_size = 400
total_batches = math.ceil(len(paper_ids) / batch_size)

for i in range(total_batches):
    chunk = paper_ids[i * batch_size : (i + 1) * batch_size]
    print(f"batch {i+1}/{total_batches} …")

    df_chunk = fetch_refs_years(chunk)
    if df_chunk.empty:
        continue

    df_chunk = df_chunk.explode("reference_ids").dropna(subset=["reference_ids"])

    # internal edges only
    df_chunk = df_chunk[df_chunk["reference_ids"].isin(paper_set)]
    if df_chunk.empty:
        continue

    for _, row in df_chunk.iterrows():
        src = row["id"]
        tgt = row["reference_ids"]
        year_src = row["year"]

        edges.append((src, tgt))

        if src not in neighbors:
            neighbors[src] = []
        neighbors[src].append([tgt, id2year.get(tgt)])

    time.sleep(1.0)

# =====================================================
# SAVE OUTPUT
# =====================================================

edges_df = pd.DataFrame(edges, columns=["source", "target"])
edges_df.to_csv("citation_edges_internal.csv", index=False)

neighbors_df = pd.DataFrame([
    {"id": pid, "neighbors": neighbors.get(pid, [])}
    for pid in paper_ids
])
neighbors_df.to_json("citation_neighbors_internal.json", orient="records", indent=2)

print(f"Saved {len(edges_df)} internal edges → citation_edges_internal.csv")
print(f"Saved neighbor lists → citation_neighbors_internal.json")



Using default endpoint: 'https://app.dimensions.ai'


5740 ids loaded
Dimcli - Dimensions API Client (v1.4)
Connected to: <https://app.dimensions.ai/api/dsl> - DSL v2.13
Method: manual login


Starting iteration with limit=400 skip=0 ...


batch 1/15 …


0-400 / 400 (6.32s)
400-400 / 400 (5.82s)
===
Records extracted: 400
Starting iteration with limit=400 skip=0 ...


...

batch 14/15 …


0-400 / 400 (3.38s)
400-400 / 400 (0.43s)
===
Records extracted: 400
Starting iteration with limit=400 skip=0 ...


batch 15/15 …


0-140 / 140 (0.60s)
===
Records extracted: 140


Saved 76086 internal edges → citation_edges_internal.csv
Saved neighbor lists → citation_neighbors_internal.json


In [14]:
import pandas as pd
import json

# =====================================================
# LOAD INTERNAL EDGES
# =====================================================

edges_df = pd.read_csv("citation_edges_internal.csv")

# load paper → year mapping
id2year = dict(zip(df["id"], df["year"]))

# =====================================================
# BUILD INCOMING EDGE LIST (invert graph)
# =====================================================

incoming = {}   # incoming[target] = [sources]

for src, tgt in edges_df.itertuples(index=False):
    if tgt not in incoming:
        incoming[tgt] = []
    incoming[tgt].append(src)

# =====================================================
# COMPUTE CITATION TIMELINES
# =====================================================

citation_timeline = {}  # id -> {year: cumulative count}

for tgt, sources in incoming.items():
    # citing years
    years = [id2year.get(src) for src in sources if id2year.get(src) is not None]
    if not years:
        continue

    # count per year
    yearly_counts = {}
    for y in years:
        yearly_counts[y] = yearly_counts.get(y, 0) + 1

    # cumulative sorted
    cumulative = {}
    total = 0
    for y in sorted(yearly_counts):
        total += yearly_counts[y]
        cumulative[y] = total

    citation_timeline[tgt] = cumulative

# =====================================================
# SAVE RESULTS
# =====================================================

# JSON
with open("citation_growth_over_time.json", "w") as f:
    json.dump(citation_timeline, f, indent=2)

# Flat CSV
rows = []
for pid, timeline in citation_timeline.items():
    for year, count in timeline.items():
        rows.append([pid, year, count])

df_out = pd.DataFrame(rows, columns=["id", "year", "cumulative_citations"])
df_out.to_csv("citation_growth_over_time.csv", index=False)

print("Saved citation growth timeline → citation_growth_over_time.csv")
print("Saved JSON version → citation_growth_over_time.json")


Saved citation growth timeline → citation_growth_over_time.csv
Saved JSON version → citation_growth_over_time.json
